# Imports

In [4]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from rdkit import Chem
import json
import time
import requests
import re

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths_and_constants import Paths, Constants
_P = Paths()
_C = Constants()

# Functions

In [86]:
import json
import time
from pathlib import Path
import requests

def sequence_getter(
    uniprot_ids,
    checkpoint_path="uniprot_checkpoint.json",
    save_every=50,
    timeout=15,
    sleep_s=0.0,
    retry_failed=True,
    show_progress=True,
):
    """
    Download UniProt sequences with checkpoint resume.
    Fetch order:
    1) direct FASTA by accession
    2) direct JSON by accession
    3) UniProt search fallback (accession/id/secondary accession)
    Returns:
      done   -> {input_id: sequence}
      failed -> {input_id: error_message}
    """
    ckpt = Path(checkpoint_path)

    if ckpt.exists():
        with ckpt.open("r", encoding="utf-8") as f:
            state = json.load(f)
        done = state.get("done", {})
        failed = state.get("failed", {})
    else:
        done, failed = {}, {}

    ids = []
    seen = set()
    for x in uniprot_ids:
        uid = str(x).strip()
        if uid and uid.lower() != "nan" and uid not in seen:
            ids.append(uid)
            seen.add(uid)

    if retry_failed:
        for uid in ids:
            failed.pop(uid, None)

    pending = [uid for uid in ids if uid not in done and uid not in failed]

    total = len(ids)
    pending_total = len(pending)
    overall_done = total - pending_total

    def save_ckpt():
        tmp = ckpt.with_suffix(ckpt.suffix + ".tmp")
        with tmp.open("w", encoding="utf-8") as f:
            json.dump({"done": done, "failed": failed}, f, ensure_ascii=False, indent=2)
        tmp.replace(ckpt)

    if show_progress:
        print(
            f"Resume status -> done: {len(done)}, failed: {len(failed)}, "
            f"pending: {pending_total}, total: {total}"
        )

    with requests.Session() as session:
        for i, uid in enumerate(pending, start=1):
            try:
                seq = ""

                # 1) Direct FASTA
                r = session.get(f"https://rest.uniprot.org/uniprotkb/{uid}.fasta", timeout=timeout)
                if r.status_code == 200:
                    lines = r.text.strip().splitlines()
                    if lines and lines[0].startswith(">"):
                        seq = "".join(lines[1:]).replace(" ", "").upper()

                # 2) Direct JSON
                if not seq:
                    rj = session.get(f"https://rest.uniprot.org/uniprotkb/{uid}.json", timeout=timeout)
                    if rj.status_code == 200:
                        seq = rj.json().get("sequence", {}).get("value", "").upper()

                # 3) Search fallback (covers secondary accessions / renamed entries)
                if not seq:
                    rs = session.get(
                        "https://rest.uniprot.org/uniprotkb/search",
                        params={
                            "query": f"(accession:{uid}) OR (id:{uid}) OR (sec_acc:{uid})",
                            "format": "json",
                            "size": 5,
                        },
                        timeout=timeout,
                    )
                    rs.raise_for_status()
                    results = rs.json().get("results", [])

                    chosen = None
                    for entry in results:
                        acc = entry.get("primaryAccession", "")
                        kb_id = entry.get("uniProtkbId", "")
                        sec = set(entry.get("secondaryAccessions", []))
                        if uid == acc or uid == kb_id or uid in sec:
                            chosen = entry
                            break
                    if chosen is None and results:
                        chosen = results[0]

                    if chosen:
                        seq = chosen.get("sequence", {}).get("value", "").upper()

                if not seq:
                    raise ValueError(f"No sequence found for: {uid}")

                done[uid] = seq
                failed.pop(uid, None)

            except Exception as e:
                failed[uid] = str(e)

            if i % save_every == 0:
                save_ckpt()

            if show_progress and (i % save_every == 0 or i == pending_total):
                print(
                    f"Progress pending: {i}/{pending_total} ({(i/pending_total*100 if pending_total else 100):.2f}%) | "
                    f"overall: {overall_done + i}/{total} ({((overall_done + i)/total*100 if total else 100):.2f}%)"
                )

            if sleep_s > 0:
                time.sleep(sleep_s)

    save_ckpt()
    return done, failed

MUT_RE = re.compile(r"^([A-Z])(\d+)([A-Z])$")

def parse_substitution(sub):
    """
    Convert 'A123G' -> ('A', 123, 'G')
    """
    m = MUT_RE.match(sub.strip().upper())
    if not m:
        raise ValueError(f"Substitution format invalid: {sub}")
    wt, pos, mut = m.groups()
    return wt, int(pos), mut

def apply_substitutions(seq, substitutions, sep=r"[;,]\s*"):
    """
    Applies one or multiple substitutions to a sequence.
    - seq: original sequence (string)
    - substitutions: 'A123G' o 'A123G;L130P'
    - sep: separator between multiple substitutions
    """
    seq_list = list(seq)
    muts = re.split(sep, substitutions.strip())

    for sub in muts:
        if not sub:
            continue
        wt, pos, mut = parse_substitution(sub)
        idx = pos - 1  # 1-based -> 0-based

        if idx < 0 or idx >= len(seq_list):
            raise IndexError(f"Position out of range: {pos} for sequence of length {len(seq_list)}")

        if seq_list[idx] != wt:
            raise ValueError(
                f"Mismatch in {sub}: expected '{wt}' at position {pos}, "
                f"but found '{seq_list[idx]}'"
            )

        seq_list[idx] = mut

    return "".join(seq_list)

def peek_mutations(wt_seq, mut_seq, ignore_case=True):
    """
    Compares wt_seq vs mut_seq y devuelve una lista de mutaciones tipo A23L.
    """
    if ignore_case:
        wt_seq = wt_seq.upper()
        mut_seq = mut_seq.upper()

    if len(wt_seq) != len(mut_seq):
        raise ValueError(
            f"Sequences have different lengths: {len(wt_seq)} vs {len(mut_seq)}"
        )

    muts = []
    for i, (wt_aa, mut_aa) in enumerate(zip(wt_seq, mut_seq), start=1):  # 1-based
        if wt_aa != mut_aa:
            muts.append(f"{wt_aa}{i}{mut_aa}")
    return muts

def pdb_id_getter_from_smiles(
    smiles_iterable,
    checkpoint_path="pdb_ccd_checkpoint.json",
    save_every=50,
    timeout=20,
    sleep_s=0.0,
    retry_failed=True,
    show_progress=True,
):
    """
    Busca PDB 3-letter IDs (CCD) por SMILES usando InChIKey exact_match.

    Checkpoint JSON:
      done[canonical_smiles] = {
        "molecule_id": str | None,      # se asigna candidates[0] si hay candidatos
        "status": "unique" | "ambiguous_assigned_first" | "no_match" | "invalid_smiles",
        "candidates": [str, ...],
        "inchikey": str | None
      }
      failed[canonical_smiles_o_raw] = "error"

    Retorna:
      done, failed
    """
    ckpt = Path(checkpoint_path)

    if ckpt.exists():
        with ckpt.open("r", encoding="utf-8") as f:
            state = json.load(f)
        done = state.get("done", {})
        failed = state.get("failed", {})
    else:
        done, failed = {}, {}

    # Normaliza + dedup por canonical SMILES (si es válido)
    keys = []
    seen = set()
    raw_to_key = {}

    for x in smiles_iterable:
        raw = "" if x is None else str(x).strip()
        if not raw or raw.lower() == "nan":
            key = raw
        else:
            mol = Chem.MolFromSmiles(raw)
            key = Chem.MolToSmiles(mol, canonical=True) if mol is not None else raw

        if key not in seen:
            keys.append(key)
            seen.add(key)
        raw_to_key[raw] = key

    if retry_failed:
        for k in keys:
            failed.pop(k, None)

    pending = [k for k in keys if k not in done and k not in failed]

    def save_ckpt():
        tmp = ckpt.with_suffix(ckpt.suffix + ".tmp")
        with tmp.open("w", encoding="utf-8") as f:
            json.dump({"done": done, "failed": failed}, f, ensure_ascii=False, indent=2)
        tmp.replace(ckpt)

    if show_progress:
        print(f"Resume -> done: {len(done)}, failed: {len(failed)}, pending: {len(pending)}, total: {len(keys)}")

    with requests.Session() as session:
        for i, can_smiles in enumerate(pending, start=1):
            try:
                mol = Chem.MolFromSmiles(can_smiles) if can_smiles else None
                if mol is None:
                    done[can_smiles] = {
                        "molecule_id": None,
                        "status": "invalid_smiles",
                        "candidates": [],
                        "inchikey": None,
                    }
                    continue

                inchikey = Chem.MolToInchiKey(mol)

                payload = {
                    "query": {
                        "type": "terminal",
                        "service": "text",
                        "parameters": {
                            "attribute": "rcsb_chem_comp_descriptor.InChIKey",
                            "operator": "exact_match",
                            "value": inchikey,
                        },
                    },
                    "return_type": "chem_comp",
                    "request_options": {"return_all_hits": True},
                }

                r = session.post("https://search.rcsb.org/rcsbsearch/v2/query", json=payload, timeout=timeout)
                r.raise_for_status()

                candidates = sorted({
                    x["identifier"]
                    for x in r.json().get("result_set", [])
                    if "identifier" in x
                })

                if len(candidates) == 0:
                    rec = {"molecule_id": None, "status": "no_match", "candidates": [], "inchikey": inchikey}
                elif len(candidates) == 1:
                    rec = {"molecule_id": candidates[0], "status": "unique", "candidates": candidates, "inchikey": inchikey}
                else:
                    # Asigna el primero, pero lo deja marcado como ambiguo
                    rec = {
                        "molecule_id": candidates[0],
                        "status": "ambiguous_assigned_first",
                        "candidates": candidates,
                        "inchikey": inchikey,
                    }

                done[can_smiles] = rec
                failed.pop(can_smiles, None)

            except Exception as e:
                failed[can_smiles] = str(e)

            if i % save_every == 0:
                save_ckpt()
            if show_progress and (i % save_every == 0 or i == len(pending)):
                print(f"Progress: {i}/{len(pending)}")

            if sleep_s > 0:
                time.sleep(sleep_s)

    save_ckpt()
    return done, failed


# Propertyprep for binding (biolip)

In [40]:
biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)
print(biol.columns)

biol["protein_sequence"] = biol["protein_sequence"].astype(str).str.strip()
biol["uniprot_id"] = biol["uniprot_id"].replace("", pd.NA)

seq_to_uniprot = (
    biol.dropna(subset=["uniprot_id"])
      .drop_duplicates(subset=["protein_sequence"])
      .set_index("protein_sequence")["uniprot_id"]
)

biol["uniprot_id"] = biol["uniprot_id"].fillna(biol["protein_sequence"].map(seq_to_uniprot))

Index(['chain_id', 'resolution', 'binding_site_id', 'ligand_id',
       'ligand_chain_id', 'ligand_serial_number', 'binding_site_residues',
       'nearby_residues', 'ligand_contact_residues', 'catalytic_site_residues',
       'ec_number', 'go_terms', 'binding_affinity_1', 'binding_affinity_2',
       'binding_affinity_3', 'binding_affinity_4', 'uniprot_id', 'pubmed_id',
       'binding_site_range', 'protein_sequence', 'ligand_id_raw', 'smiles',
       'ligand_name', 'map_status', 'label'],
      dtype='object')


/tmp/ipykernel_18933/1070420007.py:1: DtypeWarning: Columns (13,14,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)


In [32]:
biol['smiles'] = biol['smiles'].astype(str).str.strip()
biol['ligand_id_raw'] = biol['ligand_id_raw'].replace("", pd.NA)
biol['ligand_name'] = biol['ligand_name'].replace("", pd.NA)

smiles_to_id = (
    biol.dropna(subset=['ligand_id_raw'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_id_raw']
)

smiles_to_name = (
    biol.dropna(subset=['ligand_name'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_name']
)

biol['ligand_id_raw'] = biol['ligand_id_raw'].fillna(biol['smiles'].map(smiles_to_id))
biol['ligand_name'] = biol['ligand_name'].fillna(biol['smiles'].map(smiles_to_name))

In [42]:
final_cols = ['uniprot_id', 'protein_sequence', 'ligand_id', 'ligand_name', 'smiles', 'label']
biol_final = biol[final_cols]
biol_final = biol_final.rename(columns={'label': 'binding'})
biol_final = biol_final.reset_index(drop=True)
biol_final.to_csv(_P.data_processed_properties / 'binding' / 'biolip_binding.csv', index=False)

# Propertyprep for DDG (FireProtDB)

In [48]:
ddg = pd.read_csv(_P.data_interim_dir / 'fireprotdb' / 'fireprotdb_ddg.csv', index_col=0)
ddg = ddg.reset_index()
print(ddg.columns)

Index(['EXPERIMENT_ID', 'SEQUENCE_ID', 'MUTANT_ID', 'SOURCE_SEQUENCE_ID',
       'TARGET_SEQUENCE_ID', 'SEQUENCE_LENGTH', 'SUBSTITUTION', 'INSERTION',
       'DELETION', 'PROTEIN', 'ORGANISM', 'CHYMOTRYPSIN_ML', 'CM', 'DCP',
       'DDG', 'DG', 'DH', 'DHVH', 'DOMAINOME_DDG', 'DOMAINOME_DDG_STD',
       'DOMAINOME_FITNESS', 'DOMAINOME_FITNESS_STD', 'DTM', 'M',
       'REVERSIBILITY', 'STABILIZING', 'STATE', 'TM', 'TRYPSIN_ML', 'BUFFER',
       'BUFFER_CONC', 'EXP_TEMPERATURE', 'ION', 'ION_CONC', 'MEASURE',
       'METHOD', 'PH', 'EC_NUMBER', 'UNIPROTKB', 'INTERPRO', 'MEGASCALE',
       'ACTIVE_SITE', 'BINDING_SITE', 'CONSERVATION', 'WWPDB', 'B_FACTOR',
       'IN_TUNNEL', 'IN_POCKET', 'PUBLICATION_PMID', 'PUBLICATION_DOI',
       'PUBLICATION_YEAR', 'SOURCE_DATASET', 'REFERENCING_DATASET'],
      dtype='object')


In [49]:
fire_cols = ['EXPERIMENT_ID', 'UNIPROTKB', 'SUBSTITUTION', 'DDG']
ddg = ddg[fire_cols]
unique_prot = ddg['UNIPROTKB'].unique()
unique_prot

array(['P0A877', 'P00651', 'P69441', 'P00720', 'P61626', 'P00698',
       'P00929', 'P00644', 'P33665', 'P0A7Y4', 'P29957', 'P00648',
       'P62593', 'P29768', 'P81708', 'P06241', 'P61823', 'P05798',
       'P03706', 'Q9REI6', 'P96110', 'P04179', 'P11961', 'P22069',
       'O74035', 'P00918', 'P0AFG6', 'Q13526', 'P00782', 'P00563',
       'P0ABQ4', 'P00750', 'P00811', 'P07170', 'Q16595', 'P0C2S3',
       'P23540', 'O61594', 'P11309', 'P16304', 'O49003', 'Q8TZE8',
       'P00593', 'P07311', 'P30289', 'P07445', 'P62942', 'Q8WZ42',
       'P00509', 'P0A434', 'P37001', 'P00560', 'P18912', 'P00952',
       'O95363', 'P00558', 'P20533', 'P06213', 'P00912', 'P00807',
       'P03958', 'P00592', 'Q23551', 'P15917', 'P35670', 'P38398',
       'P41500', 'P09850', 'Q88IY3'], dtype=object)

In [50]:
done, failed = sequence_getter(
    unique_prot,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 121, failed: 0, pending: 0, total: 69


In [52]:
ddg["sequence"] = ddg['UNIPROTKB'].map(done)
ddg_f = ddg.rename(columns={
    'EXPERIMENT_ID': _C.dataset_id,
    'UNIPROTKB': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'SUBSTITUTION': 'mutation',
    'DDG': 'ddg'
})
ddg_f['mut_protein_sequence'] = ddg_f['protein_sequence'] #creating new column to then change to a mutated sequence
ddg_f['mut_protein_sequence'] = ddg_f.apply(
    lambda row: apply_substitutions(row['protein_sequence'], row['mutation']), axis=1) #changing to mutated sequence
print(ddg_f.columns)
ddg_f.rename(columns={'protein_sequence': 'wildtype_sequence', 'mut_protein_sequence': 'protein_sequence'}, inplace=True)
print(ddg_f.columns)
ddg_f = ddg_f[
    ["dataset_id", "uniprot_id", "wildtype_sequence", "mutation", "protein_sequence", "ddg"]
]
print(ddg_f.columns)
#Sanity check: peek mutations from wildtype vs mutated sequence and compare with reported mutation
a = ddg_f.loc[0, "wildtype_sequence"]
b = ddg_f.loc[0, "protein_sequence"]
print(a)
print(b)
print(peek_mutations(a, b))

ddg_f.to_csv(_P.data_processed_properties / 'ddg_mut' / 'fireprotdb_ddg.csv', index=False)

Index(['dataset_id', 'uniprot_id', 'mutation', 'ddg', 'protein_sequence',
       'mut_protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'mutation', 'ddg', 'wildtype_sequence',
       'protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'wildtype_sequence', 'mutation',
       'protein_sequence', 'ddg'],
      dtype='object')
MERYESLFAQLKERKEGAFVPFVTLGDPGIEQSLKIIDTLIEAGADALELGIPFSDPLADGPTIQNATLRAFAAGVTPAQCFEMLALIRQKHPTIPIGLLMYANLVFNKGIDEFYAQCEKVGVDSVLVADVPVEESAPFRQAALRHNVAPIFICPPNADDDLLRQIASYGRGYTYLLSRAGVTGAENRAALPLNHLVAKLKEYNAAPPLQGFGISAPDQVKAAIDAGAAGAISGSAIVKIIEQHINEPEKMLAALKVFVQPMKAATRS
MERYESLFAQLKERKEGAFVPLVTLGDPGIEQSLKIIDTLIEAGADALELGIPFSDPLADGPTIQNATLRAFAAGVTPAQCFEMLALIRQKHPTIPIGLLMYANLVFNKGIDEFYAQCEKVGVDSVLVADVPVEESAPFRQAALRHNVAPIFICPPNADDDLLRQIASYGRGYTYLLSRAGVTGAENRAALPLNHLVAKLKEYNAAPPLQGFGISAPDQVKAAIDAGAAGAISGSAIVKIIEQHINEPEKMLAALKVFVQPMKAATRS
['F22L']


# Propertyprep for dtm (FireProtDB)

In [42]:
dtm = pd.read_csv(_P.data_interim_dir / 'fireprotdb' / 'fireprotdb_dtm.csv', index_col=0)
dtm = dtm.reset_index()
print(dtm.columns)
fire_cols = ['EXPERIMENT_ID', 'UNIPROTKB', 'SUBSTITUTION', 'DTM']
dtm = dtm[fire_cols]
unique_prot_dtm = dtm['UNIPROTKB'].unique()
unique_prot_dtm

Index(['EXPERIMENT_ID', 'SEQUENCE_ID', 'MUTANT_ID', 'SOURCE_SEQUENCE_ID',
       'TARGET_SEQUENCE_ID', 'SEQUENCE_LENGTH', 'SUBSTITUTION', 'INSERTION',
       'DELETION', 'PROTEIN', 'ORGANISM', 'CHYMOTRYPSIN_ML', 'CM', 'DCP',
       'DDG', 'DG', 'DH', 'DHVH', 'DOMAINOME_DDG', 'DOMAINOME_DDG_STD',
       'DOMAINOME_FITNESS', 'DOMAINOME_FITNESS_STD', 'DTM', 'M',
       'REVERSIBILITY', 'STABILIZING', 'STATE', 'TM', 'TRYPSIN_ML', 'BUFFER',
       'BUFFER_CONC', 'EXP_TEMPERATURE', 'ION', 'ION_CONC', 'MEASURE',
       'METHOD', 'PH', 'EC_NUMBER', 'UNIPROTKB', 'INTERPRO', 'MEGASCALE',
       'ACTIVE_SITE', 'BINDING_SITE', 'CONSERVATION', 'WWPDB', 'B_FACTOR',
       'IN_TUNNEL', 'IN_POCKET', 'PUBLICATION_PMID', 'PUBLICATION_DOI',
       'PUBLICATION_YEAR', 'SOURCE_DATASET', 'REFERENCING_DATASET'],
      dtype='object')


array(['P0A877', 'P00651', 'P69441', 'P00720', 'P61626', 'P00698',
       'P00929', 'P00644', 'P61495', 'P33665', 'P00883', 'P0A7Y4',
       'P29957', 'P00648', 'P62593', 'P29768', 'P81708', 'P06241',
       'P26514', 'Q43763', 'P61823', 'P05798', 'P03706', 'P04789',
       'P41365', 'P05645', 'Q9REI6', 'P80040', 'P00942', 'P47189',
       'P96110', 'P04179', 'P11961', 'P24058', 'P22069', 'O68541',
       'P00183', 'O74035', 'P00918', 'P50455', 'Q13526', 'P00782',
       'P00563', 'P21549', 'Q9L9D7', 'P06547', 'P00445', 'P06182',
       'P0ABQ4', 'P00889', 'Q56304', 'P00750', 'P00749', 'P00441',
       'P00943', 'Q56319', 'P07170', 'P25984', 'P08038', 'P28366',
       'P10408', 'P60484', 'Q16595', 'Q5S007', 'Q5KZ26', 'P48842',
       'P14756', 'P0A115', 'D4Z2G1', 'Q9ZEU2', 'O25949', 'P23458',
       'P60953', 'P0C2S3', 'P23540', 'P18429', 'P19515', 'O61594',
       'Q8NBP7', 'P26882', 'P16304', 'O95571', 'Q9ZAG3', 'O49003',
       'P11413', 'A0A060GYS1', 'P63096', 'P39940', 'Q54276', '

In [43]:
done, failed = sequence_getter(
    unique_prot_dtm,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 121, failed: 0, pending: 0, total: 92


In [63]:
dtm["sequence"] = dtm['UNIPROTKB'].map(done)
dtm_f = dtm.rename(columns={
    'EXPERIMENT_ID': _C.dataset_id,
    'UNIPROTKB': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'SUBSTITUTION': 'mutation',
    'DTM': 'dtm'
})
print(dtm_f.shape)
dtm_f = dtm_f.dropna(subset=['mutation'])
print(dtm_f.shape)

(3514, 5)
(3509, 5)


In [64]:
dtm_f['mut_protein_sequence'] = dtm_f['protein_sequence']
dtm_f['mut_protein_sequence'] = dtm_f.apply(lambda row: apply_substitutions(row['protein_sequence'], row['mutation']), axis=1)
print(dtm_f.columns)
dtm_f.rename(columns={'protein_sequence': 'wildtype_sequence', 'mut_protein_sequence': 'protein_sequence'}, inplace=True)
print(dtm_f.columns)
dtm_f = dtm_f[
    ["dataset_id", "uniprot_id", "wildtype_sequence", "mutation", "protein_sequence", "dtm"]
]
print(dtm_f.columns)
dtm_f.to_csv(_P.data_processed_properties / 'dtm' / 'fireprotdb_dtm.csv', index=False)

Index(['dataset_id', 'uniprot_id', 'mutation', 'dtm', 'protein_sequence',
       'mut_protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'mutation', 'dtm', 'wildtype_sequence',
       'protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'wildtype_sequence', 'mutation',
       'protein_sequence', 'dtm'],
      dtype='object')


# Polypropertypreb phopt (BRENDA)

In [72]:
brenda_raw = pd.read_csv(_P.data_interim_dir / 'phopt' / 'brenda_phopt.csv', index_col=0)
print(brenda_raw.columns)
print(brenda_raw.shape)
print(len(brenda_raw['uniprot_from_brenda'].unique()))
unique_brenda = brenda_raw['uniprot_from_brenda'].unique()

Index(['protein_id', 'organism', 'uniprot_from_brenda', 'phopt_median',
       'n_measurements', 'phopt_std'],
      dtype='object')
(12369, 6)
11557


In [69]:
done, failed = sequence_getter(
    unique_brenda,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 121, failed: 0, pending: 11516, total: 11557
Progress pending: 10/11516 (0.09%) | overall: 51/11557 (0.44%)
Progress pending: 20/11516 (0.17%) | overall: 61/11557 (0.53%)
Progress pending: 30/11516 (0.26%) | overall: 71/11557 (0.61%)
Progress pending: 40/11516 (0.35%) | overall: 81/11557 (0.70%)
Progress pending: 50/11516 (0.43%) | overall: 91/11557 (0.79%)
Progress pending: 60/11516 (0.52%) | overall: 101/11557 (0.87%)
Progress pending: 70/11516 (0.61%) | overall: 111/11557 (0.96%)
Progress pending: 80/11516 (0.69%) | overall: 121/11557 (1.05%)
Progress pending: 90/11516 (0.78%) | overall: 131/11557 (1.13%)
Progress pending: 100/11516 (0.87%) | overall: 141/11557 (1.22%)
Progress pending: 110/11516 (0.96%) | overall: 151/11557 (1.31%)
Progress pending: 120/11516 (1.04%) | overall: 161/11557 (1.39%)
Progress pending: 130/11516 (1.13%) | overall: 171/11557 (1.48%)
Progress pending: 140/11516 (1.22%) | overall: 181/11557 (1.57%)
Progress pending: 150/11516 (1.30%) 

In [75]:
cols = ['uniprot_from_brenda', 'phopt_median','n_measurements', 'phopt_std']
brenda_opt = brenda_raw[cols]
brenda_opt["sequence"] = brenda_opt['uniprot_from_brenda'].map(done)
brenda_opt = brenda_opt.rename(columns={
    'uniprot_from_brenda': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'phopt': 'phopt'
})
print(brenda_opt.shape)
brenda_opt = brenda_opt[[_C.uniprot_id, _C.protein_sequence, 'phopt_median', 'n_measurements', 'phopt_std']]
brenda_opt.to_csv(_P.data_processed_properties / 'ph_opt' / 'brenda_phopt.csv', index=False)

(12369, 5)


/tmp/ipykernel_26795/1190427124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brenda_opt["sequence"] = brenda_opt['uniprot_from_brenda'].map(done)


# Propertyprep for kcat (CatPred)

In [89]:
kcat_raw = pd.read_csv(_P.data_raw_catpred_dir / 'CatPred_kcat.csv', index_col=0)
kcat_columns = ['uniprot', 'sequence', 'reactant_smiles', 'temperature', 'ph', 'log10kcat_max']
kcat = kcat_raw[kcat_columns]
kcat = kcat.rename(columns={
    'uniprot': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'reactant_smiles': _C.smiles,
    'temperature': 'temperature',
    'ph': 'ph',
    'log10kcat_max': 'kcat'
})
kcat.to_csv(_P.data_processed_properties / 'kcat' / 'catpred_kcat.csv', index=False)

# Propertyprep for km (CatPred)

# Propertyprep for EC number (UniProt)

# Propertyprep for active sites (UniProt)